# Monte Carlo Permutation Tests — Overfit Detection
Referencia: Bailey et al. (2014); De Prado (2018) Cap. 11

---
## 1 · Config

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid

import McOverfit as mc
from McOverfit.cache import list_cached
from cpcv_analysis.data import load_data

# ── config ────────────────────────────────────────────────────────────────
RANDOM_SEED    = 42
N_PERMUTATIONS = 100
IS_FRAC        = 0.70
N_JOBS         = mc.DEFAULT_N_JOBS   # 50% de cores por defecto

print(f"N_JOBS = {N_JOBS}  (de {os.cpu_count()} cores)")
list_cached()

---
## 2 · Datos + split + grid search real

In [ ]:
X, y, t1, prices, fwd_ret = load_data()

cut      = int(len(X) * IS_FRAC)
X_is     = X.iloc[:cut];  y_is  = y.iloc[:cut];  fwd_is  = fwd_ret.iloc[:cut].values
X_oos    = X.iloc[cut:];  y_oos = y.iloc[cut:];  fwd_oos = fwd_ret.iloc[cut:].values
dates_is  = X.index[:cut]
dates_oos = X.index[cut:]

PARAM_GRID = list(ParameterGrid({
    "n_estimators":    [50, 100, 200],
    "max_depth":       [3, 5, None],
    "min_samples_leaf": [5, 20],
}))

# grid search real sobre IS
from sklearn.ensemble import RandomForestClassifier
best_params, best_is_sr, real_is_eq = mc.grid_search_rf(
    X_is.values, y_is.values, fwd_is, PARAM_GRID, seed=RANDOM_SEED
)

clf_real = RandomForestClassifier(**best_params, random_state=RANDOM_SEED, n_jobs=-1)
clf_real.fit(X_is.values, y_is.values)
real_oos_pnl = (2 * clf_real.predict(X_oos.values) - 1) * fwd_oos
real_oos_eq  = mc.equity_curve(real_oos_pnl)
real_oos_sr  = mc.annualized_sr(real_oos_pnl)

print(f"IS obs: {len(X_is)} | OOS obs: {len(X_oos)}")
print(f"Real IS  Sharpe: {best_is_sr:.3f}")
print(f"Real OOS Sharpe: {real_oos_sr:.3f}")

---
## 3 · Permutaciones IS  *(cacheadas)*

In [ ]:
perm_is = mc.run_is(
    X_is.values, y_is.values, fwd_is, PARAM_GRID,
    n_perm=N_PERMUTATIONS, is_frac=IS_FRAC, seed=RANDOM_SEED, n_jobs=N_JOBS
)
pvalue_is = float((perm_is["sharpes"] >= best_is_sr).mean())
print(f"p-value (IS): {pvalue_is:.4f}")

---
## 4 · Permutaciones WalkForward  *(cacheadas)*

In [ ]:
perm_wf = mc.run_wf(
    X_is.values, y_is.values, fwd_is, X_oos.values, fwd_oos, PARAM_GRID,
    n_perm=N_PERMUTATIONS, is_frac=IS_FRAC, seed=RANDOM_SEED, n_jobs=N_JOBS
)
pvalue_wf = float((perm_wf["sharpes"] >= real_oos_sr).mean())
print(f"p-value (WF): {pvalue_wf:.4f}")

---
## 5 · Plot 1 — IS Permutation Curves

In [ ]:
mc.plots.is_curves(
    dates_is, real_is_eq, perm_is, N_PERMUTATIONS,
    save_path="../plots/mc_is_permutation_curves.png"
)

---
## 6 · Plot 2 — IS Sharpe Histogram

In [ ]:
mc.plots.is_histogram(
    perm_is, best_is_sr, pvalue_is, N_PERMUTATIONS,
    save_path="../plots/mc_is_permutation_histogram.png"
)

---
## 7 · Plot 3 — WF Permutation Curves

In [ ]:
import numpy as np
log_close_is = np.log(prices["Close"].loc[dates_is])

mc.plots.wf_curves(
    dates_is, dates_oos, log_close_is, real_oos_eq, real_oos_sr,
    perm_wf, pvalue_wf, N_PERMUTATIONS,
    save_path="../plots/mc_wf_permutation_test.png"
)

---
## 8 · Plot 4 — WF OOS Sharpe Histogram

In [ ]:
mc.plots.wf_histogram(
    perm_wf, real_oos_sr, pvalue_wf, N_PERMUTATIONS,
    save_path="../plots/mc_wf_permutation_histogram.png"
)

---
## 9 · Resumen

In [ ]:
summary = pd.DataFrame({
    "Test":         ["IS Permutation", "WalkForward Permutation"],
    "Real Sharpe":  [round(best_is_sr, 3),             round(real_oos_sr, 3)],
    "Perm SR mean": [round(perm_is["sharpes"].mean(), 3), round(perm_wf["sharpes"].mean(), 3)],
    "Perm SR std":  [round(perm_is["sharpes"].std(),  3), round(perm_wf["sharpes"].std(),  3)],
    "p-value":      [round(pvalue_is, 4),              round(pvalue_wf, 4)],
    "N perm":       [N_PERMUTATIONS,                   N_PERMUTATIONS],
})
summary["Significant (p<0.05)"] = summary["p-value"] < 0.05
summary.set_index("Test", inplace=True)
print(summary.to_string())

print("\n--- Cache actual ---")
list_cached()
summary